In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# https://stackoverflow.com/questions/21971449/how-do-i-increase-the-cell-width-of-the-jupyter-ipython-notebook-in-my-browser
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import json
import pickle
import random
from IPython.display import display, Image, clear_output
from PIL import Image as PILImage
# from pigeonXT import annotate
from collections import defaultdict
import sys
import os
from glob import glob
from pprint import pprint
import pandas as pd
import utils

In [ ]:
# display code
def show_imgs_sidebyside(imgs, captions, grayscale_imgs=None, title=None, figsize=None):
    """ Assume images already loaded """
    NUM_ROWS = 1
    IMGS_PER_ROW = len(imgs)
    f, ax = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=figsize if figsize else (30,4),)# constrained_layout=True)
    
    for i, (img, caption) in enumerate(zip(imgs, captions)):
        ax[i].imshow(img, cmap='gray')
        if isinstance(caption, str):
            ax[i].set_title(caption)
        else:
            ax[i].set_title(**caption)
        ax[i].axes.xaxis.set_ticks([])
        ax[i].axes.yaxis.set_ticks([])
        if title is not None and i == 0:            
            ax[i].set_ylabel(**title, rotation=0, fontsize=14, labelpad=22)
    plt.tight_layout()
        
    if grayscale_imgs is not None:
        assert len(grayscale_imgs) == len(imgs)
        f1, ax1 = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=(16,6))
    if grayscale_imgs is not None:
        for i, img in enumerate(grayscale_imgs):
            ax1[i].imshow(img)  # cmap='gray'
            ax1[i].axes.xaxis.set_ticks([])
            ax1[i].axes.yaxis.set_ticks([])
            if title is not None and i == 0:            
                ax1[i].set_ylabel(**title, rotation=0, fontsize=10, labelpad=22)
        plt.tight_layout()
    plt.show()


In [ ]:
def load_matching_run(exp_dir: str, exp_prefix: str):
    # glob for everything after prefix
    files = sorted(glob(f"{os.path.join(exp_dir, exp_prefix)}*.csv"))
    print(f'Found {len(files)} matching files. Characters: ')
    print(', '.join([fname[-5] for fname in files]))
    data = dict()
    for fname in files:
        queries = []
        candidates = []
        sims = []
        row_paths = []
        with open(fname) as f:
            for line in f.readlines():
                line = line.strip()
#                 this_query = line.split(',')[0]
#                 this_candidates = line.split(',')[1:11]
#                 this_sims = [float(s) for s in line.split(',')[11:21]]
                queries.append(this_query)
                candidates.append(this_candidates)
                sims.append(this_sims)
                row_paths.append([this_query] + this_candidates)
        data[fname[-5]] = {'row_paths': row_paths, 'queries': queries, 'candidates': candidates, 'sims': sims}
        # with open(fname, 'rb') as f:
        #     data[fname[-5]] = pickle.load(f)
    return data

In [ ]:
data = load_matching_run(
    exp_dir='/graft2/code/nvog/git/matching/output/evaluate_ckpt/',
    # exp_prefix='manualcriticalenquiries_fnames-braddyll_plutarch-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-braddyll_kingjames-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-braddyll_critical-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-locke_two_treatises-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-locke_letter-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-forty_sermons-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-braddyll-topk_'
    # exp_prefix='manualcriticalenquiries_fnames-everingham_english-topk_'
    # exp_prefix='manualfortysermons_fnames-everingham_english-topk_'
    # exp_prefix='manualfortysermons_fnames-braddyll-topk_'
#     exp_prefix='manualfortysermons_fnames-braddyll_critical-topk_'
    # exp_prefix='manuallocke_fnames-everingham_english-topk_'
    # exp_prefix='manualspinoza_fnames-everingham_english-topk_'
    # exp_prefix='manualspinoza_fnames-everingham_anekdota-topk_'
    # exp_prefix='manualfortysermons_fnames-everingham_anekdota-topk_'
    # exp_prefix='manualfortysermons_fnames-everingham_anekdota-topk_'
#     exp_prefix='manuallocke_fnames-braddyll_critical-topk_'  # X
#     exp_prefix='manualcriticalenquiries_fnames-locke_two_treatises-topk_'  # X  
#     exp_prefix='manualcriticalenquiries_fnames-locke_letter-topk_'  # X
#     exp_prefix='manualspinoza_fnames-braddyll_critical-topk_'  # X
#     exp_prefix='manualspinoza_fnames-braddyll-topk_'  # X good matches found to religionandreasonREDO1688!
#     exp_prefix='manuallocke_fnames-braddyll-topk_'  # X good matches found to religionandreasonREDO1688 too!
#     exp_prefix='manualfortysermons_fnames-braddyll_religionandreason-topk_'  # X
#     exp_prefix='manualcriticalenquiries_fnames-braddyll_religionandreason-topk_'  # X
#     exp_prefix='manualspinoza_fnames-braddyll_religionandreason-topk_'   # great matches!!
#     exp_prefix='manuallocke_fnames-everingham_anekdota-topk_'   # X
#     exp_prefix='manualspinoza_fnames-everingham_anekdota-topk_'   # a few potential matches
#     exp_prefix='manuallocke_fnames-braddyll_twobooks-topk_'  # X not much here - probably bc they're separated by 10 years?
#     exp_prefix='manualspinoza_fnames-everingham_ogygia-topk_'  # X not really
#     exp_prefix='manuallocke_fnames-everingham_ogygia-topk_'  # not much here - some potential maybes
#     exp_prefix='manualfortysermons_fnames-everingham_ogygia-topk_'   # many good matches
#     exp_prefix='manualspinoza_fnames-everingham_ogygia-topk_'  # nothing
#     exp_prefix='manualcriticalenquiries_fnames-spinoza_theological-topk_'  # X nothing
#     exp_prefix='manuallocke_fnames-everingham_examen-topk_'  # X nothing
#     exp_prefix='manualspinoza_fnames-everingham_examen-topk_'  # X nothing
#     exp_prefix='manualfortysermons_fnames-everingham_examen-topk_'  # X some decent ones
#     exp_prefix='manualfortysermons_fnames-braddyll_twobooks-topk_'  # X nothing
#     exp_prefix='manualspinoza_fnames-braddyll_twobooks-topk_'  # X nothing
#     exp_prefix='manualfortysermons_fnames-everingham_english-topk_'
#     exp_prefix='manuallocke_fnames-C-braddyll_voyageofitaly-topk'  # X nothing besides broken F posted in slack earlier
#     exp_prefix='manuallocke_fnames-C-braddyll-topk'  # X nothing besides broken F posted in slack earlier   NOTE: ABC character rows not showing!
#     exp_prefix='manuallocke_fnames-C-everingham_english-topk'  # X nothing besides broken F posted in slack earlier  
    exp_prefix='manuallockespinozanature-lockespinozanature_candidates-topk'  # check performance of dual encoder on found locke spinoza nature matches
    
)

total_rows = sum([len(data[char]['queries']) for char in data.keys()])
print(f"Total rows: {total_rows}")
r = 0
max_r = 500
for char in data.keys():
    if char in 'FGMN':
        continue
    print(f"Character {char} has {len(data[char]['queries'])} rows:")
    for i in range(len(data[char]['row_paths'])):
        if r > max_r:
            break
        query_book_name = utils.get_book_name(Path(data[char]['queries'][i]).name)
        candidates_book_names = [utils.get_book_name(Path(p).name) for p in data[char]['candidates'][i]]
        query_page_num = utils.get_page_num(Path(data[char]['queries'][i]).name)
        candidates_page_nums = [utils.get_page_num(Path(p).name) for p in data[char]['candidates'][i]]
        query_line_num = utils.get_line_num(Path(data[char]['queries'][i]).name)
        candidates_line_nums = [utils.get_line_num(Path(p).name) for p in data[char]['candidates'][i]]
        query_char_loc = utils.get_char_loc(Path(data[char]['queries'][i]).name)
        candidates_char_locs = [utils.get_char_loc(Path(p).name) for p in data[char]['candidates'][i]]
        show_imgs_sidebyside(
            [np.array(PILImage.open(p)) for p in data[char]['row_paths'][i]],
            [f"Query {i}\n{query_book_name}\npage {query_page_num}, line {query_line_num}, loc {query_char_loc}" + "\n(Query)",] 
            + [
                f"Match {j+1}\n{candidates_book_names[j]}\npage {candidates_page_nums[j]}, line {candidates_line_nums[j]}, loc {candidates_char_locs[j]}" + 
                f"\n({data[char]['sims'][i][j]})"
                for j in range(10)
            ]
        )
        r += 1
    # show_imgs_sidebyside([
    #         np.random.rand(28,28), np.random.rand(28,28)
    #     ]
    #     , [
    #         'a', 'b'
    #     ], 
    #     grayscale_imgs=[
    #         np.random.rand(28,28), np.random.rand(28,28)
    #     ], 
    # )